# Car Insurance Claim Prediction

**Goal:** predict whether a policyholder will file a car insurance claim (`CLAIM_FLAG`) from their demographic, financial, vehicle and driving-history data.

**Dataset:** [Car Insurance Claim Data](https://www.kaggle.com/datasets/xiaomengsun/car-insurance-claim-data) — 10,302 policyholders, 27 columns.

**Workflow**
1. Load and inspect the data
2. Clean it (currency strings, `z_` prefixes, invalid values, missing values)
3. Exploratory data analysis
4. Encode categorical features
5. Train/test split and scaling
6. Models: Logistic Regression → class-balanced Logistic Regression → Random Forest
7. Decision-threshold tuning
8. Feature importance
9. Conclusions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report, roc_auc_score,
                             recall_score, precision_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)

sns.set_theme(style='whitegrid')

## 1. Load and inspect the data

In [ ]:
data = pd.read_csv('/kaggle/input/datasets/xiaomengsun/car-insurance-claim-data/car_insurance_claim.csv')
print(data.shape)
data.head()

In [ ]:
data.dtypes

## 2. Data cleaning

- `ID` and `BIRTH` carry no predictive information (`AGE` already captures birth date) → dropped.
- Money columns are stored as text like `$67,349` → converted to floats.
- Several categories carry a `z_` prefix (e.g. `z_No`, `z_SUV`) → prefix removed.

In [ ]:
data = data.drop(columns=['ID', 'BIRTH'])

money_cols = ['INCOME', 'HOME_VAL', 'BLUEBOOK', 'OLDCLAIM', 'CLM_AMT']
for col in money_cols:
    data[col] = data[col].str.replace('$', '', regex=False)
    data[col] = data[col].str.replace(',', '', regex=False)
    data[col] = data[col].astype(float)

for col in data.select_dtypes('object').columns:
    data[col] = data[col].str.replace('z_', '', regex=False)

for col in ['MSTATUS', 'GENDER', 'EDUCATION', 'CAR_TYPE', 'URBANICITY']:
    print(col, data[col].unique())

### Missing and invalid values

In [ ]:
data.isnull().sum()[data.isnull().sum() > 0]

In [ ]:
print('Minimum CAR_AGE:', data['CAR_AGE'].min())   # negative car age is impossible

# Only 7 rows miss AGE -> drop them
data = data.dropna(subset=['AGE'])

# Negative CAR_AGE -> treat as missing, then fill with the median
data.loc[data['CAR_AGE'] < 0, 'CAR_AGE'] = np.nan
data['CAR_AGE'] = data['CAR_AGE'].fillna(data['CAR_AGE'].median())

# Skewed numeric columns -> median imputation
for col in ['YOJ', 'INCOME', 'HOME_VAL']:
    data[col] = data[col].fillna(data[col].median())

# Missing occupation becomes its own category
data['OCCUPATION'] = data['OCCUPATION'].fillna('Unknown')

print('Missing values left:', data.isnull().sum().sum())
print(data.shape)

## 3. Exploratory data analysis

In [ ]:
print(data['CLAIM_FLAG'].value_counts(normalize=True))

ax = data['CLAIM_FLAG'].value_counts().plot(kind='bar', color=['#4C72B0', '#DD8452'])
ax.set_xticklabels(['No claim (0)', 'Claim (1)'], rotation=0)
ax.set_title('Target distribution')
plt.show()

About **27%** of policyholders filed a claim, so the classes are imbalanced. Accuracy alone will be misleading — recall on the claim class and ROC-AUC matter more.

In [ ]:
cat_features = ['URBANICITY', 'CAR_USE', 'REVOKED', 'RED_CAR', 'CAR_TYPE', 'EDUCATION']
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, cat_features):
    rate = data.groupby(col)['CLAIM_FLAG'].mean().sort_values()
    rate.plot(kind='barh', ax=ax, color='#4C72B0')
    ax.set_title(f'Claim rate by {col}')
    ax.set_xlabel('Claim rate')
plt.tight_layout()
plt.show()

In [ ]:
num_features = ['MVR_PTS', 'CLM_FREQ', 'INCOME', 'AGE', 'TRAVTIME', 'BLUEBOOK']
print(data.groupby('CLAIM_FLAG')[num_features].mean().T.round(2))

plt.figure(figsize=(12, 9))
sns.heatmap(data.select_dtypes('number').corr(), cmap='coolwarm', center=0)
plt.title('Correlation between numeric features')
plt.show()

**EDA takeaways**
- **Urban drivers claim ~5× more often** than rural drivers (~32% vs ~6%).
- **Commercial vehicles** claim more than private ones (~35% vs ~22%).
- A **revoked licence** almost doubles the claim rate (~45% vs ~24%).
- **Red cars do not claim more** (~26% vs ~27%) — the myth doesn't hold here.
- Claimants have more motor-vehicle record points, more past claims and slightly lower income.

`RED_CAR` adds nothing, and `CLM_AMT` is the claim amount itself (only known *after* a claim → target leakage), so both are dropped.

In [ ]:
data = data.drop(columns=['RED_CAR', 'CLM_AMT'])
data.shape

## 4. Encoding

In [ ]:
for col in ['PARENT1', 'MSTATUS', 'REVOKED']:
    data[col] = data[col].map({'Yes': 1, 'No': 0})

data['GENDER'] = data['GENDER'].map({'M': 1, 'F': 0})
data['CAR_USE'] = data['CAR_USE'].map({'Private': 1, 'Commercial': 0})
data['URBANICITY'] = data['URBANICITY'].map({'Highly Urban/ Urban': 1, 'Highly Rural/ Rural': 0})

# Education is ordinal
data['EDUCATION'] = data['EDUCATION'].map({'<High School': 0, 'High School': 1,
                                           'Bachelors': 2, 'Masters': 3, 'PhD': 4})

# Nominal features -> one-hot
data = pd.get_dummies(data, columns=['OCCUPATION', 'CAR_TYPE'], drop_first=True, dtype=int)

print('Missing values after encoding:', data.isnull().sum().sum())
print(data.shape)

## 5. Train/test split and scaling

In [ ]:
X = data.drop(columns=['CLAIM_FLAG'])
Y = data['CLAIM_FLAG']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)
print(X_train.shape, X_test.shape)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)   # transform only, never fit on test

## 6. Models

### 6.1 Baseline Logistic Regression

In [ ]:
logistic = LogisticRegression(max_iter=1000)
logistic.fit(X_train_scaled, Y_train)
y_pred = logistic.predict(X_test_scaled)

print('Accuracy:', round(accuracy_score(Y_test, y_pred), 3))
print('ROC-AUC :', round(roc_auc_score(Y_test, logistic.predict_proba(X_test_scaled)[:, 1]), 3))
print(classification_report(Y_test, y_pred))

79% accuracy looks fine, but **recall on claims is only ~0.41** — the model misses most actual claims because of the class imbalance.

### 6.2 Class-balanced Logistic Regression

In [ ]:
logistic_bal = LogisticRegression(max_iter=1000, class_weight='balanced')
logistic_bal.fit(X_train_scaled, Y_train)
y_pred_bal = logistic_bal.predict(X_test_scaled)

print('Accuracy:', round(accuracy_score(Y_test, y_pred_bal), 3))
print('ROC-AUC :', round(roc_auc_score(Y_test, logistic_bal.predict_proba(X_test_scaled)[:, 1]), 3))
print(classification_report(Y_test, y_pred_bal))

Balancing the classes lifts claim recall to **~0.74** at the cost of some accuracy — a good trade for an insurer, where missing a risky customer is the expensive mistake.

### 6.3 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, Y_train)   # trees don't need scaled features
rf_proba = rf.predict_proba(X_test)[:, 1]

print('Accuracy:', round(accuracy_score(Y_test, rf.predict(X_test)), 3))
print('ROC-AUC :', round(roc_auc_score(Y_test, rf_proba), 3))
print(classification_report(Y_test, rf.predict(X_test)))

The forest has the best ROC-AUC, but at the default 0.5 cutoff its claim recall is low. Its ranking is good — the cutoff is the problem.

## 7. Decision-threshold tuning (Random Forest)

In [ ]:
rows = []
for t in [0.5, 0.45, 0.4, 0.35, 0.3, 0.25, 0.2]:
    pred = (rf_proba >= t).astype(int)
    rows.append({'threshold': t,
                 'recall': recall_score(Y_test, pred),
                 'precision': precision_score(Y_test, pred),
                 'f1': f1_score(Y_test, pred),
                 'accuracy': accuracy_score(Y_test, pred)})
thresholds = pd.DataFrame(rows).round(3)
thresholds

In [ ]:
# Choose the threshold with the best F1 for the claim class
best_t = thresholds.loc[thresholds['f1'].idxmax(), 'threshold']
print('Chosen threshold:', best_t)

rf_pred_tuned = (rf_proba >= best_t).astype(int)
print(classification_report(Y_test, rf_pred_tuned))

ConfusionMatrixDisplay(confusion_matrix(Y_test, rf_pred_tuned),
                       display_labels=['No claim', 'Claim']).plot(cmap='Blues')
plt.title(f'Random Forest (threshold = {best_t})')
plt.show()

## 8. Model comparison

In [ ]:
lb_proba = logistic_bal.predict_proba(X_test_scaled)[:, 1]
lr_proba = logistic.predict_proba(X_test_scaled)[:, 1]

def summary(name, proba, t):
    pred = (proba >= t).astype(int)
    return {'model': name, 'threshold': t,
            'ROC-AUC': roc_auc_score(Y_test, proba),
            'accuracy': accuracy_score(Y_test, pred),
            'claim recall': recall_score(Y_test, pred),
            'claim precision': precision_score(Y_test, pred),
            'claim F1': f1_score(Y_test, pred)}

results = pd.DataFrame([
    summary('Logistic Regression', lr_proba, 0.5),
    summary('Logistic Regression (balanced)', lb_proba, 0.5),
    summary('Random Forest (balanced)', rf_proba, 0.5),
    summary('Random Forest (tuned threshold)', rf_proba, best_t),
]).round(3)
results

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(Y_test, lr_proba, name='Logistic', ax=ax)
RocCurveDisplay.from_predictions(Y_test, lb_proba, name='Logistic (balanced)', ax=ax)
RocCurveDisplay.from_predictions(Y_test, rf_proba, name='Random Forest', ax=ax)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_title('ROC curves')
plt.show()

## 9. Feature importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importances.head(15).sort_values().plot(kind='barh', figsize=(8, 6), color='#4C72B0')
plt.title('Top 15 features (Random Forest)')
plt.show()
importances.head(10)

In [ ]:
coefs = pd.Series(logistic_bal.coef_[0], index=X_train.columns).sort_values()
print('Pushes towards a claim:')
print(coefs.tail(8)[::-1].round(3))
print('\nPushes away from a claim:')
print(coefs.head(8).round(3))

## 10. Conclusions

- **Data:** 10,295 policyholders after cleaning; 27% filed a claim.
- **Strongest risk signals:** living in an urban area, commercial vehicle use, past claims (`CLM_FREQ`, `OLDCLAIM`), motor-vehicle record points (`MVR_PTS`), revoked licence, being a single parent, and lower income/home value. Car colour doesn't matter.
- **Models:** all three rank risk similarly well (ROC-AUC ≈ 0.80–0.81). The default logistic model misses ~60% of claims; class balancing and threshold tuning fix this.
- **Recommendation:** use the **Random Forest with a tuned threshold** (or the balanced logistic model if interpretability matters more). See the comparison table above for exact numbers.

**Next steps:** gradient boosting (XGBoost / LightGBM), cross-validated hyper-parameter search, and calibrated probabilities for pricing.